# GRACE H3-v2 — Bắt đầu full pipeline local

Branch: **feat/h3-v2-benchmark**. MCC là chính, F1 thứ hai; cấu hình mặc định Devign + ReVeal, seed42.

Nếu đang khôi phục validation cũ bị Vertex chặn, chạy notebooks/retry-blocked-validation.ipynb trước, rồi khởi động lại kernel riêng của entry này và chạy từ Bước 1/2. Recovery trộn Qwen/Vertex và DeepSeek/FPT; kết quả chỉ dùng exploratory.

| Lượt | Việc làm | Tiếp tục ở đâu |
|---|---|---|
| 1 | Entry tạo base-inputs local | Chạy colab_cpu_prepare.ipynb trên máy, PHASE='base' |
| 2 | Entry nhập base-output, chạy GRACE validation | Chạy lại entry; sau đó runner local, PHASE='meta' |
| 3 | Entry nhập meta-output, chạy GRACE test và đánh giá | Xem benchmark_summary.json trong runs/h3v2/ |

Notebook CPU giữ tên cũ để không mất đường dẫn, nhưng chạy hoàn toàn local: không Colab, Drive, upload hay download. Mọi zip input/output/checkpoint nằm trong bundles/.

CPU stages chạy offline khi dependencies và Java21 đã cài sẵn. Vertex LLM cần mạng trừ khi exact response cache đủ mọi request. RUN_INFERENCE=True bật gọi API; False chỉ dùng cache.

H3-v2 chưa có benchmark. Test không dùng để chọn winner.

**Profile hiện tại: validation 8.192 / test 8.192 byte UTF-8.** Snapshot kết quả test 24k đã được commit trước khi đổi profile. Việc trộn Qwen/Vertex và DeepSeek/FPT trong validation vẫn được ghi là exploratory.


In [6]:
from pathlib import Path
import os, sys, json, subprocess
candidates=[Path.cwd(),Path.cwd().parent,Path('D:/JUNIOR/LabThayTho/GRACE-H3-BenchmarkEnsemble'),Path('/mnt/d/JUNIOR/LabThayTho/GRACE-H3-BenchmarkEnsemble')]
if os.environ.get('GRACE_H3V2_ROOT'): candidates.insert(0,Path(os.environ['GRACE_H3V2_ROOT']))
ROOT=next((p.resolve() for p in candidates if (p/'configs/benchmark_v2.json').exists()),None)
if ROOT is None: raise RuntimeError('Mở notebook từ worktree GRACE-H3-BenchmarkEnsemble hoặc đặt GRACE_H3V2_ROOT')
sys.path.insert(0,str(ROOT))
INSTALL_DEPENDENCIES=False
PREPARE_IN_LOCAL_NOTEBOOK=True
RUN_INFERENCE=True
TEST_CONTEXT_LIMIT_BYTES=8192
BUNDLES=ROOT/'bundles'
BUNDLES.mkdir(exist_ok=True)
print('ROOT:',ROOT)
print('Entry local của branch benchmark; CPU runner chạy trên cùng máy.')


ROOT: /mnt/d/JUNIOR/LabThayTho/GRACE-H3-BenchmarkEnsemble
Entry local của branch benchmark; CPU runner chạy trên cùng máy.


In [7]:
if INSTALL_DEPENDENCIES:
    subprocess.check_call([sys.executable,'-m','pip','install','--break-system-packages','-r',str(ROOT/'requirements-v2.txt'),'-r',str(ROOT/'requirements.txt')])
from grace_h1.io import load_local_environment
load_local_environment(ROOT)
from grace_h3v2 import workflow as w
from grace_h3v2.portable import create_bundle,import_bundle,identity
import zipfile
import pandas as pd
CFG=w.config(ROOT)
import importlib.util
_context_spec=importlib.util.spec_from_file_location('h3_test_context_profile', ROOT/'tools/test_context_profile.py')
test_context=importlib.util.module_from_spec(_context_spec)
_context_spec.loader.exec_module(test_context)
print(json.dumps(CFG,indent=2))
def show_stage(name):
    try: return w.load_stage(ROOT,CFG,name)
    except FileNotFoundError: return None
def cpu_handoff(phase,selection):
    result=show_stage(selection)
    if result is not None:
        print(f'Using local {selection}; validating existing artifacts, skipping output ZIP import.')
        return w.base_prepare(ROOT,CFG) if phase=='base' else w.meta_prepare(ROOT,CFG)
    output=BUNDLES/f'grace-h3v2-{phase}-output.zip'
    if output.exists(): print(import_bundle(ROOT,output,'output',phase))
    result=show_stage(selection)
    if result is not None: return result
    if not PREPARE_IN_LOCAL_NOTEBOOK:
        return w.base_prepare(ROOT,CFG) if phase=='base' else w.meta_prepare(ROOT,CFG)
    archive=BUNDLES/f'grace-h3v2-{phase}-inputs.zip'
    reuse=False
    if archive.exists():
        with zipfile.ZipFile(archive) as z:
            m=json.loads(z.read('bundle_manifest.json'))
            reuse=m.get('job')==identity(ROOT) and m.get('phase')==phase and m.get('kind')=='inputs'
    if not reuse: print(create_bundle(ROOT,archive,'inputs',phase))
    raise RuntimeError(f'DỪNG: mở notebooks/colab_cpu_prepare.ipynb trên máy local, đặt PHASE={phase!r} và Run All. Input: {archive}. Output sẽ ghi vào {output}. Sau đó quay lại notebook này và Run All.')


{
  "schema": "h3v2_benchmark_config_v1",
  "datasets": [
    "devign",
    "reveal"
  ],
  "seeds": [
    42
  ],
  "folds": 3,
  "inner_folds": 3,
  "meta_folds": 3,
  "group_key": "code_sha256",
  "linear_C": 1,
  "selection": {
    "primary": "mcc",
    "secondary": "f1",
    "tie_tolerance": 0.001,
    "cost_penalty": 0
  },
  "threshold_grid": [
    0,
    0.01,
    0.02,
    0.03,
    0.04,
    0.05,
    0.06,
    0.07,
    0.08,
    0.09,
    0.1,
    0.11,
    0.12,
    0.13,
    0.14,
    0.15,
    0.16,
    0.17,
    0.18,
    0.19,
    0.2,
    0.21,
    0.22,
    0.23,
    0.24,
    0.25,
    0.26,
    0.27,
    0.28,
    0.29,
    0.3,
    0.31,
    0.32,
    0.33,
    0.34,
    0.35,
    0.36,
    0.37,
    0.38,
    0.39,
    0.4,
    0.41,
    0.42,
    0.43,
    0.44,
    0.45,
    0.46,
    0.47,
    0.48,
    0.49,
    0.5,
    0.51,
    0.52,
    0.53,
    0.54,
    0.55,
    0.56,
    0.57,
    0.58,
    0.59,
    0.6,
    0.61,
    0.62,
    0.63,
    0.64,
    0

## Nạp lại cấu hình FPT từ `.env`

Chạy riêng cell dưới sau khi sửa `D:/JUNIOR/LabThayTho/.env` (WSL: `/mnt/d/JUNIOR/LabThayTho/.env`), trước cell inference validation hoặc test. Không cần restart kernel hay chạy lại bước CPU. Cell nạp `FPT_API`, `FPT_MODEL`, `FPT_BASE_URL` và chỉ hiển thị model/endpoint; không gọi API.

Inference cũng đọc lại file này khi bắt đầu chạy: các biến dưới dùng để kiểm tra cấu hình hiện tại, không khóa cấu hình nếu bạn sửa `.env` tiếp. Đổi model hoặc endpoint sẽ chọn cache request tương ứng với cấu hình mới.


In [ ]:
# Chạy lại cell này bất cứ lúc nào sau khi thay đổi shared .env.
from grace_h1.fpt_provider import load_fpt_settings

_fpt_settings = load_fpt_settings()  # Cùng parser và đường dẫn với inference.
if _fpt_settings is None:
    raise FileNotFoundError('Không tìm thấy shared LabThayTho/.env')
if not _fpt_settings['api_key']:
    raise ValueError('FPT_API đang trống trong shared .env')

FPT_API = _fpt_settings['api_key']
FPT_MODEL = _fpt_settings['model']
FPT_BASE_URL = _fpt_settings['base_url'].rstrip('/')
for _name, _value in {
    'FPT_API': FPT_API,
    'FPT_MODEL': FPT_MODEL,
    'FPT_BASE_URL': FPT_BASE_URL,
}.items():
    os.environ[_name] = _value
# Không để biến vòng lặp giữ thêm bản sao API key.
del _name, _value, _fpt_settings

print('Đã nạp lại shared LabThayTho/.env')
print('FPT_MODEL:', FPT_MODEL)
print('FPT_BASE_URL:', FPT_BASE_URL)
print('FPT_API: đã nạp (ẩn)')


## Bước 1 — CPU local: classifier và chuẩn bị validation

Entry tạo base-inputs rồi dừng. Chạy notebooks/colab_cpu_prepare.ipynb local với PHASE='base'. CPU fit classifier bằng nested train OOF, chọn gate bằng MCC/F1, chuẩn bị prompts validation và ghi base-output vào bundles/. CodeT5 frozen; CPU runner không gọi LLM.


In [8]:
base=w.base_prepare(ROOT,CFG) if not PREPARE_IN_LOCAL_NOTEBOOK else cpu_handoff('base','base_selection')
print(pd.DataFrame([{'dataset':j['dataset'],'seed':j['seed'],'base_folder':j['base_folder']} for j in base['jobs']]))


Using local base_selection; validating existing artifacts, skipping output ZIP import.
  dataset  seed                                        base_folder
0  devign    42  artifacts/h3v2/base/devign/42/f6d28bb5a30eed5a...
1  reveal    42  artifacts/h3v2/base/reveal/42/9e5b6bc3bb5e930e...


## Bước 2 — Local: GRACE validation ngoài test

Preflight và gọi các request còn thiếu tại local; exact cache được reuse. API cần mạng. Khi validation complete, Run All tạo meta-inputs local cho lượt CPU tiếp theo.

Nếu validation cũ có partial_request_errors, chuyển sang retry-blocked-validation.ipynb, chạy đến cell gộp, rồi quay lại bước này. Không cần gọi lại các response Vertex thành công.


In [9]:
plans=w.preflight(ROOT,CFG,'validation')
print(json.dumps(plans,indent=2))
reports=w.infer(ROOT,CFG,'validation',allow_api=RUN_INFERENCE)
print(json.dumps(reports,indent=2))
if any(r['status']!='complete' for r in reports): raise RuntimeError('Validation chưa complete. Nếu đang dùng recovery trộn model, chạy notebooks/retry-blocked-validation.ipynb đến bước gộp rồi chạy lại cell này; xem reports để biết request còn lỗi.')


[
  {
    "run": "runs/devign/validation/baseline_whitening/f79c31718971e534",
    "status": "complete",
    "new_unique_requests": 0,
    "mixed_model_recovery": true
  },
  {
    "run": "runs/reveal/validation/baseline_whitening/27ee63ea9ec23f93",
    "status": "complete",
    "new_unique_requests": 0,
    "mixed_model_recovery": true
  }
]
[
  {
    "run": "runs/devign/validation/baseline_whitening/f79c31718971e534",
    "status": "complete",
    "reused_completed_run": true,
    "api_attempts_this_run": 0,
    "new_unique_requests": 0,
    "stored_inference_summary": "runs/devign/validation/baseline_whitening/f79c31718971e534/inference_summary.json"
  },
  {
    "run": "runs/reveal/validation/baseline_whitening/27ee63ea9ec23f93",
    "status": "complete",
    "reused_completed_run": true,
    "api_attempts_this_run": 0,
    "new_unique_requests": 0,
    "stored_inference_summary": "runs/reveal/validation/baseline_whitening/27ee63ea9ec23f93/inference_summary.json"
  }
]


## Bước 3 — CPU local: meta selector/stacking và test features

Sau validation complete, entry tạo meta-inputs trong bundles/. Chạy colab_cpu_prepare.ipynb local với PHASE='meta'. CPU fit router/selector/stacking trên development, khóa winner, tính test features không nhãn, và xuất meta-output vào bundles/. Không gọi LLM ở CPU runner.

Nhánh benefit-router chỉ dùng feature trước expert; selector/stacking dùng expert output. Full-reference/stacking có thể cần100% query: benchmark mode.


In [10]:
meta=w.meta_prepare(ROOT,CFG) if not PREPARE_IN_LOCAL_NOTEBOOK else cpu_handoff('meta','meta_selection')
print(pd.DataFrame([{'dataset':j['dataset'],'seed':j['seed'],'winner_development':j['development_winner']} for j in meta['jobs']]))


Using local meta_selection; validating existing artifacts, skipping output ZIP import.
  dataset  seed winner_development
0  devign    42         gate_tuned
1  reveal    42         gate_tuned


## Bước 4 — Local: test inference và đánh giá

Winner đã khóa từ development. Local preflight/test-infer reuse exact cache hoặc gọi Vertex nếu RUN_INFERENCE=True. Test scores đã tính ở CPU runner local; nhãn test chỉ mở trong evaluate.

Test mặc định dùng **8.192 byte UTF-8**, bằng validation. Bước này chỉ dựng lại prompt từ retrieval/graph cache, không fit lại model hoặc chạy Joern. Run test 24k và bảng điểm cũ được lưu riêng; response chỉ reuse khi prompt/model/spec khớp hoàn toàn. Sau khi đổi profile, chưa có bảng điểm 8k mới cho tới khi cell inference/evaluate hoàn tất.


In [ ]:
profile=test_context.prepare(ROOT,CFG,TEST_CONTEXT_LIMIT_BYTES)
print(json.dumps(profile,indent=2))
print(json.dumps(w.preflight(ROOT,CFG,'test'),indent=2))
reports=w.infer(ROOT,CFG,'test',allow_api=RUN_INFERENCE)
print(json.dumps(reports,indent=2))
if any(r['status']!='complete' for r in reports): raise RuntimeError('Test requests chưa complete; resume inference trước evaluate.')
w.predict(ROOT,CFG)
summary=w.evaluate(ROOT,CFG)
summary=test_context.annotate_summary(ROOT,CFG,summary)


In [ ]:
table=[]
for run in summary['datasets']:
    for arm,result in run['arms'].items():
        m=result['metrics']
        table.append({'dataset':run['dataset'],'seed':run['seed'],'arm':arm,'locked_winner':arm==run['locked_development_winner'],'MCC':m['mcc'],'F1':m['f1'],'precision':m['precision'],'recall':m['recall'],'specificity':m['specificity'],'invalid':m['invalid_n'],'expert_query_ratio':result['algorithmic_expert_query_ratio'],'fallback':result['fallback_n']})
display(pd.DataFrame(table))
print('Summary:',w.stage_file(ROOT,CFG,'benchmark_summary'))
print('Không chọn lại winner dựa trên bảng test. So với gate_tuned và full_grace; chi phí routing lý thuyết tách actual API/cache.')
